# 📊 Task 1 — Notebook 1: Data Analysis
## PneumoniaMNIST Exploratory Data Analysis

**Purpose:** Understand the dataset before training — class distribution, pixel statistics, augmentation effects.

**Author:** [Your Name] | **GitHub:** [Your Repo URL]

---
### Notebooks in this project:
| Notebook | Purpose |
|----------|---------|
| `01_data_analysis.ipynb` | ← **You are here** — EDA |
| `02_train.ipynb` | Model training |
| `03_evaluate.ipynb` | Evaluation & comparison |

## 0. Setup

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install medmnist scikit-learn seaborn matplotlib torchvision --quiet

# ── Clone repo & add to path ───────────────────────────────────────────────
import os
REPO_URL  = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # ← UPDATE THIS
REPO_NAME = 'YOUR_REPO'                                        # ← UPDATE THIS

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

os.chdir(REPO_NAME)
import sys; sys.path.insert(0, '.')
print('Repo ready. CWD:', os.getcwd())

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
from medmnist import PneumoniaMNIST

# ── Import from our data module ────────────────────────────────────────────
from data.dataset import (
    get_dataloaders, get_datasets, get_class_weights,
    CLASS_NAMES, MEAN, STD, INFO_DATA
)

OUTPUT_DIR = 'outputs/data_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Imports OK')

ModuleNotFoundError: No module named 'medmnist'

In [3]:
#!pip install medmnist

## 1. Dataset Overview

In [ ]:
print('=== PneumoniaMNIST Dataset Info ===')
for k, v in INFO_DATA.items():
    print(f'  {k}: {v}')

print(f'\nClass names: {CLASS_NAMES}')

# Load all splits
train_ds, val_ds, test_ds = get_datasets(download=True)
print(f'\nSplit sizes:')
print(f'  Train: {len(train_ds)}')
print(f'  Val:   {len(val_ds)}')
print(f'  Test:  {len(test_ds)}')
print(f'  Total: {len(train_ds)+len(val_ds)+len(test_ds)}')

## 2. Class Distribution

In [ ]:
# Count per split
def count_classes(ds):
    labels = [int(ds[i][1]) for i in range(len(ds))]
    return {'Normal': labels.count(0), 'Pneumonia': labels.count(1), 'total': len(labels)}

splits      = {'Train': train_ds, 'Val': val_ds, 'Test': test_ds}
split_stats = {name: count_classes(ds) for name, ds in splits.items()}

for split, stats in split_stats.items():
    n, p, t = stats['Normal'], stats['Pneumonia'], stats['total']
    print(f'{split:6s}: Normal={n} ({n/t*100:.1f}%)  Pneumonia={p} ({p/t*100:.1f}%)')

# Plot
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = ['#4CAF50', '#F44336']

for ax, (split, stats) in zip(axes, split_stats.items()):
    vals   = [stats['Normal'], stats['Pneumonia']]
    wedges, texts, autotexts = ax.pie(
        vals, labels=CLASS_NAMES, colors=colors,
        autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2)
    )
    for t in autotexts: t.set_fontsize(11)
    ax.set_title(f'{split} set\n(n={stats["total"]})', fontsize=12, fontweight='bold')

plt.suptitle('Class Distribution across Splits', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

pos_weight, n_neg, n_pos = get_class_weights()
print(f'\nClass imbalance → pos_weight for BCELoss = {pos_weight:.3f}')

## 3. Sample Images Grid

In [ ]:
# Raw images (no augmentation) for visualization
raw_ds = PneumoniaMNIST(split='train', transform=transforms.ToTensor(),
                         download=True, size=28)
labels_all    = [int(raw_ds[i][1]) for i in range(len(raw_ds))]
normal_idx    = [i for i, l in enumerate(labels_all) if l == 0][:10]
pneumonia_idx = [i for i, l in enumerate(labels_all) if l == 1][:10]

fig, axes = plt.subplots(2, 10, figsize=(20, 4.5))

for col, idx in enumerate(normal_idx):
    axes[0, col].imshow(raw_ds[idx][0].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[0, col].axis('off')
axes[0, 0].set_ylabel('Normal', fontsize=12, fontweight='bold', color='#4CAF50')

for col, idx in enumerate(pneumonia_idx):
    axes[1, col].imshow(raw_ds[idx][0].squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[1, col].axis('off')
axes[1, 0].set_ylabel('Pneumonia', fontsize=12, fontweight='bold', color='#F44336')

plt.suptitle('Sample Images — PneumoniaMNIST (28×28 px)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Pixel Intensity Statistics

In [ ]:
# Compute pixel-level stats per class
normal_pixels, pneumonia_pixels = [], []

for i in range(len(raw_ds)):
    img, lbl = raw_ds[i]
    pixels = img.numpy().flatten()
    if int(lbl) == 0:
        normal_pixels.extend(pixels.tolist())
    else:
        pneumonia_pixels.extend(pixels.tolist())

normal_pixels    = np.array(normal_pixels)
pneumonia_pixels = np.array(pneumonia_pixels)

print('Pixel statistics (raw [0,1] range):')
print(f'  Normal    → mean={normal_pixels.mean():.3f}  std={normal_pixels.std():.3f}  '
      f'min={normal_pixels.min():.3f}  max={normal_pixels.max():.3f}')
print(f'  Pneumonia → mean={pneumonia_pixels.mean():.3f}  std={pneumonia_pixels.std():.3f}  '
      f'min={pneumonia_pixels.min():.3f}  max={pneumonia_pixels.max():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
bins = np.linspace(0, 1, 50)
axes[0].hist(normal_pixels,    bins=bins, alpha=0.6, color='#4CAF50',
             label='Normal',    density=True)
axes[0].hist(pneumonia_pixels, bins=bins, alpha=0.6, color='#F44336',
             label='Pneumonia', density=True)
axes[0].set_xlabel('Pixel Intensity'); axes[0].set_ylabel('Density')
axes[0].set_title('Pixel Intensity Distribution', fontsize=12, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Mean image per class
normal_mean    = np.mean([raw_ds[i][0].squeeze().numpy()
                           for i, l in enumerate(labels_all) if l == 0], axis=0)
pneumonia_mean = np.mean([raw_ds[i][0].squeeze().numpy()
                           for i, l in enumerate(labels_all) if l == 1], axis=0)

axes[1].imshow(np.hstack([normal_mean, pneumonia_mean]), cmap='gray')
axes[1].set_title('Mean Image: Normal (left) vs Pneumonia (right)',
                  fontsize=12, fontweight='bold')
axes[1].axis('off')
axes[1].axvline(28, color='white', lw=2)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/pixel_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Data Augmentation Visualization

In [ ]:
from data.dataset import get_transforms

aug_transform = get_transforms('train')

# Pick one normal and one pneumonia sample
sample_normal    = raw_ds[normal_idx[0]][0]
sample_pneumonia = raw_ds[pneumonia_idx[0]][0]

# Convert to PIL for augmentation pipeline
from torchvision.transforms.functional import to_pil_image

fig, axes = plt.subplots(4, 8, figsize=(18, 9))
titles = ['Normal augmentations (top 2 rows)', 'Pneumonia augmentations (bottom 2 rows)']

for row, (sample, name) in enumerate([
        (sample_normal, 'Normal'), (sample_pneumonia, 'Pneumonia')]):
    pil = to_pil_image(sample)
    for col in range(8):
        aug_img = aug_transform(pil)   # apply augmentation
        # denorm back for display
        display = (aug_img * 0.5 + 0.5).squeeze().clamp(0,1).numpy()
        r = row * 2 + (col // 4)
        c = col % 4 + (row % 1) * 0  # layout: 2 rows × 4 cols per class
        axes[row*2][col].imshow(display, cmap='gray', vmin=0, vmax=1)
        axes[row*2][col].axis('off')
        if col == 0:
            axes[row*2][0].set_ylabel(name, fontsize=11,
                fontweight='bold',
                color='#4CAF50' if name=='Normal' else '#F44336')

    # Second row: more augmented versions
    for col in range(8):
        aug_img = aug_transform(pil)
        display = (aug_img * 0.5 + 0.5).squeeze().clamp(0,1).numpy()
        axes[row*2+1][col].imshow(display, cmap='gray', vmin=0, vmax=1)
        axes[row*2+1][col].axis('off')

plt.suptitle(
    'Augmentation Examples: H-Flip + Rotation(±10°) + ColorJitter\n'
    '(Each cell is an independently augmented version of the same image)',
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary

In [ ]:
print('=== DATA ANALYSIS SUMMARY ===')
print(f'Dataset:        PneumoniaMNIST (MedMNIST v2)')
print(f'Task:           Binary classification — Normal vs Pneumonia')
print(f'Image size:     28×28 grayscale')
print(f'Train:          {len(train_ds)} images')
print(f'Val:            {len(val_ds)} images')
print(f'Test:           {len(test_ds)} images')
print(f'Class imbalance: Normal={n_neg} ({n_neg/(n_neg+n_pos)*100:.1f}%) | '
      f'Pneumonia={n_pos} ({n_pos/(n_neg+n_pos)*100:.1f}%)')
print(f'pos_weight:     {pos_weight:.3f} (used in BCEWithLogitsLoss)')
print(f'Outputs saved:  {OUTPUT_DIR}/')
print('\n→ Next: Open 02_train.ipynb to train models')